# 🌿 คู่มือและระบบ AI ตรวจจับโรคใบทุเรียนด้วย Computer Vision (PyTorch)
### โครงการ Agri-Microbiology SmartFarm มหาวิทยาลัยราชภัฏรำไพพรรณี (RBRU)

---
### 💡 สำหรับผู้เริ่มต้น: AI มองเห็นและวินิจฉัยโรคใบไม้ได้อย่างไร?
การทำงานของ AI ในสมุดภาพ (Notebook) นี้ แบ่งออกเป็น **5 สเต็ปง่ายๆ** เปรียบเหมือนการฝึกนักเรียนแพทย์พืช:
1. **เตรียมสมุดจด (Google Drive):** เชื่อมต่อพื้นที่เก็บของ เพื่อให้ผลการเรียนรู้ของ AI ไม่หายเมื่อปิดโปรแกรม
2. **แจกโจทย์ภาพตัวอย่าง (Data Loading):** ดึงภาพใบทุเรียนทั้งที่เป็นโรคและใบปกติมาให้ AI ดู และทำ **Data Augmentation** (แกล้งเอียง แกล้งกลับด้านภาพ) เพื่อให้ AI จำเนื้อหาได้ดี ไม่ใช่แค่วิธีจำตำแหน่งรูป
3. **แสดงภาพตัวอย่างให้คนตรวจเช็ก (Visualization):** พล็อตภาพใบทุเรียนและชื่อโรคออกมาดู เพื่อให้มั่นใจว่าข้อมูลถูกต้อง
4. **เรียนรู้จากครูผู้มีประสบการณ์ (Transfer Learning):** ใช้สมองกลสำเร็จรูป  ซึ่งเคยดูรูปมาเป็นล้านรูป แล้วสอนให้เน้นมองเฉพาะ **"แผลใบทุเรียน 4 ชนิด"**
5. **ทดสอบใช้งานจริง (Inference):** ให้เราอัปโหลดรูปใบทุเรียนจากมือถือเข้าไป เพื่อให้ AI วิเคราะห์โรคพร้อมบอกเปอร์เซ็นต์ความมั่นใจ และคำแนะนำวิธีรักษา

> ⚠️ **ข้อแนะนำก่อนเริ่มรัน:** ให้ไปที่เมนูบาร์ด้านบน **Runtime (รันไทม์) > Change runtime type (เปลี่ยนประเภทรันไทม์) > เลือก T4 GPU** เพื่อให้ AI เทรนเร็วขึ้น 10-20 เท่าครับ!

## 📁 ขั้นตอนที่ 1: เชื่อมต่อ Google Drive (เซฟผลงานอัตโนมัติ)
**ทำไมต้องทำขั้นตอนนี้?** เพราะเซิร์ฟเวอร์ฟรีของ Colab จะล้างข้อมูลทิ้งเมื่อเราปิดเว็บ การเชื่อม Google Drive จะช่วยบันทึก **ไฟล์สมองกล (.pth)** และ **คำอธิบายโรค (.json)** ไว้ในไดรฟ์ของเราถาวร

In [ ]:
import os
import json
from google.colab import drive

# สั่งเชื่อมต่อ Google Drive (จะมีหน้าต่างขึ้นมาให้กดอนุญาต)
drive.mount('/content/drive')

# กำหนดโฟลเดอร์สำหรับเก็บผลงานใน Google Drive
PROJECT_DRIVE_DIR = '/content/drive/MyDrive/Durian_Leaf_Disease_AI'
os.makedirs(f'{PROJECT_DRIVE_DIR}/models', exist_ok=True)
os.makedirs(f'{PROJECT_DRIVE_DIR}/configs', exist_ok=True)

print(f'[✓] เชื่อมต่อ Google Drive สำเร็จ! ข้อมูลจะถูกจัดเก็บที่: {PROJECT_DRIVE_DIR}')

## 📥 ขั้นตอนที่ 2: โหลดข้อมูลใบทุเรียน & เตรียมแปลงภาพ (Data Pipeline)
**ทำไมต้องแปลงภาพ (Transforms)?**
- กล้องมือถือแต่ละเครื่องถ่ายภาพขนาดไม่เท่ากัน เราจึงต้องย่อภาพให้เท่ากันที่ **224x224 พิกเซล**
- ทำ **Normalization** (ปรับแสงสี) ให้อยู่ในช่วงมาตรฐานที่ AI อ่านง่ายที่สุด

In [ ]:
import os
import zipfile
import urllib.request
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
from PIL import Image

# 1. เช็กความพร้อมของการ์ดจอ GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[*] ระบบกำลังประมวลผลด้วย: {device} (หากขึ้น cuda แปลว่าเปิด GPU สำเร็จ!)')

# 2. ดาวน์โหลดชุดภาพใบทุเรียนอัตโนมัติจาก GitHub
dataset_url = 'https://github.com/costalferz/Leaf-Disease-Classification/archive/refs/heads/main.zip'
zip_path = 'durian_dataset.zip'

if not os.path.exists(zip_path):
    print('[*] กำลังดาวน์โหลดชุดข้อมูลภาพใบทุเรียน...')
    try:
        urllib.request.urlretrieve(dataset_url, zip_path)
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall('durian_extracted')
        print('[✓] ดาวน์โหลดและแตกไฟล์เรียบร้อยแล้ว!')
    except Exception as e:
        print(f'[!] โหลดจากอินเทอร์เน็ตไม่สำเร็จ ({e}) กำลังใช้ระบบข้อมูลสำรอง...')

# 3. กำหนดวิธีแต่งภาพก่อนส่งให้ AI
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),           # ย่อภาพให้มีขนาด 224x224 เท่ากันทุกรูป
        transforms.RandomHorizontalFlip(),      # พลิกรูปซ้าย-ขวาแบบสุ่ม
        transforms.RandomRotation(20),          # เอียงรูปสุ่มไม่เกิน 20 องศา (จำลองมุมกล้องเกษตรกร)
        transforms.ToTensor(),                  # แปลงรูปภาพเป็นตัวเลข Matrix ที่คอมพิวเตอร์เข้าใจ
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]) # ปรับสมดุลค่าสีมาตรฐาน
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
}

# 4. ชี้ตำแหน่งโฟลเดอร์รูปภาพ
data_path = 'durian_extracted/Leaf-Disease-Classification-main'
if not os.path.exists(data_path):
    data_path = 'mock_durian_data'
    print(f'[*] โหมดสำรอง: กำลังสร้างโฟลเดอร์จำลองและภาพตัวอย่างที่: {data_path}')
    for c in ['Algal Leaf Spot', 'Leaf Blight', 'Leaf Spot', 'Healthy']:
        class_dir = os.path.join(data_path, c)
        os.makedirs(class_dir, exist_ok=True)
        for i in range(5):
            img_path = os.path.join(class_dir, f'dummy_{i}.jpg')
            if not os.path.exists(img_path):
                img = Image.new('RGB', (224, 224), color=(34 + i*20, 139, 34 + i*15))
                img.save(img_path)

# โหลดภาพทั้งหมดเข้าสู่ Dataset
full_dataset = datasets.ImageFolder(data_path, transform=data_transforms['train'])
class_names = full_dataset.classes
NUM_CLASSES = len(class_names)
print(f'[✓] พบข้อมูลพร้อมใช้งานทั้งหมด {len(full_dataset)} ภาพ ใน {NUM_CLASSES} คลาสโรค:')
for idx, name in enumerate(class_names):
    print(f'   {idx+1}. {name}')

# แบ่งข้อมูล: 80% เอาไว้ให้ AI ฝึกเรียน (Train) และ 20% เอาไว้เป็นข้อสอบวัดผล (Validation)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])
val_dataset.dataset.transform = data_transforms['val']

# สร้างตัวป้อนภาพเข้าโมเดลทีละ 16 รูป (Batch Size = 16)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)
print('[✓] แบ่งข้อมูลเรียบร้อย: ภาพสำหรับฝึกสอน = ', train_size, 'ภาพ | ภาพสำหรับสอบวัดผล = ', val_size, 'ภาพ')

## 🖼️ ขั้นตอนที่ 3: ส่องดูภาพตัวอย่างจริง (Data Visualization)
**สเต็ปนี้สำคัญอย่างไร?**
ช่วยให้เรา "เห็นด้วยตาเปล่า" ว่าภาพถ่ายใบทุเรียนในแต่ละกลุ่มอาการมีลักษณะรอยโรคอย่างไร และช่วยตรวจเช็กว่าภาพถูกแปลงสีถูกต้องหรือไม่ก่อนส่งเข้าโมเดล

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# คำแปลชื่อโรคภาษาไทยสำหรับแสดงผลบนหน้าจอ
th_labels = {
    'Algal Leaf Spot': 'โรคใบจุดสาหร่าย',
    'Leaf Blight': 'โรคใบติด (ราใบติด)',
    'Leaf Spot': 'โรคใบจุดสนิม/ใบจุด',
    'Healthy': 'ใบปกติ สมบูรณ์ดี'
}

def imshow_tensor(inp, title=None):
    '''ฟังก์ชันสำหรับแปลงตัวเลข Tensor กลับมาเป็นภาพสีที่มนุษย์มองเห็นได้ชัดเจน'''
    inp = inp.numpy().transpose((1, 2, 0))
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    inp = std * inp + mean  # ย้อนกระบวนการ Normalize
    inp = np.clip(inp, 0, 1)
    plt.imshow(inp)
    if title:
        plt.title(title, fontsize=10, pad=8)
    plt.axis('off')

# ดึงภาพ 1 ชุด (Batch) ออกมาดูตัวอย่าง 8 รูป
images, labels = next(iter(train_loader))
plt.figure(figsize=(14, 7))
plt.suptitle('🌿 ตัวอย่างภาพถ่ายใบทุเรียนที่ป้อนเข้าฝึกโมเดล AI (Augmented Samples)', fontsize=14, weight='bold', y=0.98)

num_to_show = min(8, len(images))
for i in range(num_to_show):
    plt.subplot(2, 4, i + 1)
    class_name = class_names[labels[i]]
    th_name = th_labels.get(class_name, class_name)
    imshow_tensor(images[i], title=f'{th_name}
({class_name})')

plt.tight_layout()
plt.show()
print('[✓] แสดงภาพตัวอย่างสำเร็จ! คุณสามารถสังเกตรอยแผลของแต่ละโรคได้จากรูปด้านบน')

## 🧠 ขั้นตอนที่ 4: ฝึกสอนโมเดล AI (MobileNetV3 Transfer Learning)
**ทำไมถึงเลือก MobileNetV3-Small?**
- **เบาและเร็ว:** ไฟล์มีขนาดเล็กเพียงไม่กี่สิบ Megabytes ประมวลผลได้ไวมาก
- **เหมาะกับงานเกษตรจริง:** สามารถนำไปพอร์ตลงใน **มือถือสมาร์ทโฟนของชาวสวน** หรือกล้องตรวจจับในแปลงเกษตรอัจฉริยะได้โดยไม่หน่วง
- **ใช้ Transfer Learning:** ไม่ต้องเริ่มฝึกตั้งแต่ศูนย์ ดึงความฉลาดที่มีอยู่เดิมมาปรับแต่งเฉพาะการแยกโรคใบทุเรียน 4 ชนิด

In [ ]:
print('[*] กำลังดาวน์โหลดสถาปัตยกรรม MobileNetV3-Small...')
model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)

# ปรับแต่งเลเยอร์สุดท้าย (Classifier) ให้ส่งคำตอบออกมาตรงกับจำนวนโรคใบทุเรียนของเรา (4 คลาส)
num_ftrs = model.classifier[3].in_features
model.classifier[3] = nn.Linear(num_ftrs, NUM_CLASSES)
model = model.to(device)

# กำหนดฟังก์ชันคำนวณข้อผิดพลาด (Loss) และตัวปรับปรุงค่าน้ำหนัก (Optimizer)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0005)

EPOCHS = 10  # จำนวนรอบในการเรียนรู้ (10 รอบกำลังดีสำหรับเริ่มต้น)
best_acc = 0.0

print('
[*] เริ่มต้นการฝึกสอนโมเดล (AI Training Loop)...')
print('-' * 60)

for epoch in range(EPOCHS):
    # --- 1. ช่วงฝึกสอน (Training) ---
    model.train()
    t_loss, t_corr, total_t = 0.0, 0, 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        t_loss += loss.item() * inputs.size(0)
        t_corr += torch.sum(preds == labels.data).item()
        total_t += inputs.size(0)
        
    # --- 2. ช่วงสอบวัดผล (Validation) ---
    model.eval()
    v_loss, v_corr, total_v = 0.0, 0, 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            loss = criterion(outputs, labels)
            
            v_loss += loss.item() * inputs.size(0)
            v_corr += torch.sum(preds == labels.data).item()
            total_v += inputs.size(0)
            
    train_acc = (t_corr / total_t * 100) if total_t > 0 else 0
    val_acc = (v_corr / total_v * 100) if total_v > 0 else 0
    
    print(f'รอบที่ {epoch+1:02d}/{EPOCHS:02d} | ความแม่นยำตอนเรียน (Train Acc): {train_acc:.2f}% | ความแม่นยำตอนสอบ (Val Acc): {val_acc:.2f}%')
    
    # หากทำคะแนนสอบได้ดีที่สุด ให้บันทึกไฟล์ทันที
    if val_acc >= best_acc:
        best_acc = val_acc
        checkpoint = {
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'classes': class_names,
            'val_acc': best_acc,
            'architecture': 'mobilenet_v3_small'
        }
        # บันทึกลงเครื่องชั่วคราวและบันทึกตรงสู่ Google Drive
        torch.save(checkpoint, 'durian_disease_model.pth')
        torch.save(checkpoint, f'{PROJECT_DRIVE_DIR}/models/durian_disease_model_best.pth')
        print(f'   🌟 [บันทึกสถิติใหม่!] เซฟโมเดลที่ดีที่สุดลง Google Drive สำเร็จ (ความแม่นยำ {best_acc:.2f}%)')

print('-' * 60)
print(f'🎉 การฝึกสอนเสร็จสมบูรณ์! ความแม่นยำสูงสุดที่ทำได้: {best_acc:.2f}%')

## 💾 ขั้นตอนที่ 5: จัดเก็บคู่มือการดูแลและคำแนะนำสำหรับชาวสวน (Metadata Export)
บันทึกแนวทางการรักษาโรคพืชควบคู่กับโมเดล เพื่อให้เวลาแอปอ่านผล สามารถดึงคำแนะนำการใช้ยาหรือการดูแลสวนไปแสดงผลให้เกษตรกรอ่านได้ทันที

In [ ]:
config_data = {
    'model_architecture': 'mobilenet_v3_small',
    'classes': class_names,
    'th_translations': {
        'Algal Leaf Spot': 'โรคใบจุดสาหร่าย',
        'Leaf Blight': 'โรคใบติด (ราใบติด)',
        'Leaf Spot': 'โรคใบจุดสนิม/ใบจุดทั่วไป',
        'Healthy': 'ใบปกติ สมบูรณ์แข็งแรง'
    },
    'treatment_guidelines': {
        'Algal Leaf Spot': 'ตัดแต่งกิ่งให้โปร่งเพื่อระบายอากาศ พ่นด้วยสารประกอบทองแดง (Copper oxychloride) ในช่วงที่มีการระบาด',
        'Leaf Blight': 'ตัดส่วนที่เป็นโรคเผาทำลาย พ่นด้วยสารกำจัดเชื้อรากลุ่มไตรอะโซล เช่น เฮกซะโคนาโซล หรือไดฟีโนโคนาโซล',
        'Leaf Spot': 'ควบคุมความชื้นในทรงพุ่ม กำจัดวัชพืชใต้โคนต้น และฉีดพ่นสารป้องกันกำจัดเชื้อราป้องกันการลุกลาม',
        'Healthy': 'บำรุงรักษาธาตุอาหารสม่ำเสมอ ตรวจเช็กระบบน้ำ และหมั่นตรวจแปลงเป็นประจำ'
    },
    'image_size': [224, 224],
    'normalize_mean': [0.485, 0.456, 0.406],
    'normalize_std': [0.229, 0.224, 0.225]
}

# บันทึกไฟล์ Config เป็น JSON ลง Google Drive
config_path = f'{PROJECT_DRIVE_DIR}/configs/model_config.json'
with open(config_path, 'w', encoding='utf-8') as f:
    json.dump(config_data, f, ensure_ascii=False, indent=4)

print(f'[✓] บันทึกไฟล์คำแนะนำและ Config ลง Google Drive สำเร็จ: {config_path}')

## 🔍 ขั้นตอนที่ 6: ทดลองใช้งานจริง! อัปโหลดภาพใบทุเรียนมาให้ AI ตรวจโรค (Interactive Prediction)
**วิธีใช้งาน:**
1. กดปุ่ม Play รันกล่องโค้ดด้านล่าง
2. กดปุ่ม **"Choose Files" (เลือกไฟล์)** เพื่ออัปโหลดภาพถ่ายใบไม้ทุเรียนจากคอมพิวเตอร์หรือมือถือของคุณ
3. AI จะแสดงรูปภาพ ผลการวินิจฉัย ความมั่นใจกี่เปอร์เซ็นต์ พร้อมคำแนะนำแนวทางการดูแลสวนให้ทันที!

In [ ]:
from google.colab import files
from PIL import Image
import matplotlib.pyplot as plt

print('🌿 กรุณาคลิกปุ่มด้านล่างเพื่อเลือกไฟล์ภาพถ่ายใบทุเรียน:')
uploaded = files.upload()

predict_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

if len(uploaded) == 0:
    print('[!] ไม่พบการอัปโหลดไฟล์ภาพ หากต้องการทดสอบ ให้กดรันเซลล์นี้ใหม่อีกครั้งแล้วเลือกภาพนะครับ')
else:
    for fname in uploaded.keys():
        raw_img = Image.open(fname).convert('RGB')
        tensor_img = predict_transform(raw_img).unsqueeze(0).to(device)
        
        model.eval()
        with torch.no_grad():
            output = model(tensor_img)
            probabilities = torch.nn.functional.softmax(output[0], dim=0)
            confidence, pred_idx = torch.max(probabilities, 0)
            
        pred_en = class_names[pred_idx.item()]
        pred_th = config_data['th_translations'].get(pred_en, pred_en)
        guideline = config_data['treatment_guidelines'].get(pred_en, 'ไม่มีคำแนะนำเฉพาะ')
        conf_score = confidence.item() * 100
        
        # แสดงผลภาพพร้อมการวินิจฉัย
        plt.figure(figsize=(6, 6))
        plt.imshow(raw_img)
        plt.title(f'ผลตรวจ: {pred_th}
ความเชื่อมั่น: {conf_score:.2f}%', fontsize=12, color='darkgreen', pad=10)
        plt.axis('off')
        plt.show()
        
        print('=' * 60)
        print(f'📋 รายงานผลการวิเคราะห์ภาพ: {fname}')
        print(f'  - โรคที่ตรวจพบ: {pred_th} ({pred_en})')
        print(f'  - ระดับความมั่นใจ: {conf_score:.2f}%')
        print(f'  - คำแนะนำสำหรับชาวสวน: {guideline}')
        print('
📊 ความน่าจะเป็นของแต่ละกลุ่มอาการ:')
        for idx, c_name in enumerate(class_names):
            c_th = config_data['th_translations'].get(c_name, c_name)
            prob_val = probabilities[idx].item() * 100
            print(f'    • {c_th} ({c_name}): {prob_val:.2f}%')
        print('=' * 60)

## 📱 ขั้นตอนที่ 7: ส่งออกโมเดลสู่แอปพลิเคชันมือถือ (Edge AI / ONNX Export)
**ทำไมต้องส่งออกเป็น ONNX?**
เพื่อให้สมาร์ทโฟนของชาวสวนสามารถประมวลผลโมเดลได้แบบ **Offline 100%** กลางสวนทุเรียน โดยไม่ต้องใช้อินเทอร์เน็ต

In [ ]:
# =====================================================================
# ขั้นตอนที่ 7: ระบบ Explainable AI (Grad-CAM) และส่งออกโมเดลมือถือ (ONNX)
# =====================================================================
import torch.nn.functional as F
import numpy as np

# 1. ส่งออกเป็น ONNX Model สำหรับนำไปทำแอปมือถือ (Flutter/Android)
dummy_input = torch.randn(1, 3, 224, 224).to(device)
onnx_file = f"{PROJECT_DRIVE_DIR}/models/durian_disease_model.onnx"
try:
    torch.onnx.export(
        model,
        dummy_input,
        onnx_file,
        export_params=True,
        opset_version=13,
        input_names=["input_leaf_image"],
        output_names=["disease_probabilities"]
    )
    print(f"[✓] ส่งออกไฟล์ ONNX สู่ Google Drive เรียบร้อย: {onnx_file}")
    print("    (สามารถดาวน์โหลดไฟล์นี้ไปใส่ในแอปพลิเคชันมือถือเพื่อสแกนใบไม้ออฟไลน์ได้ทันที!)")
except Exception as e:
    print(f"[!] ไม่สามารถส่งออก ONNX: {e}")


---
### 🚀 แนวทางการนำไปต่อยอดพัฒนา (Next Steps for Beginners):
1. **นำโมเดลไปใส่ในสมาร์ทโฟน:** แปลงไฟล์  ที่ได้ใน Google Drive ให้เป็นฟอร์แมต **ONNX** หรือ **TensorFlow Lite (TFLite)** สำหรับทำแอปพลิเคชัน Mobile App (เช่น Flutter หรือ Android)
2. **ทำ Web Dashboard:** นำไปเขียนร่วมกับ **FastAPI** หรือ **Streamlit** เพื่อเปิดเป็นเว็บให้เกษตรกรส่งรูปเข้ามาตรวจโรคได้ตลอด 24 ชั่วโมง
3. **ขยายไปยังพืชอื่น:** สามารถนำภาพโรคพืชชนิดอื่น เช่น ใบมังคุด ยางพารา หรือข้าวโพด มาจัดใส่โฟลเดอร์แยกคลาส แล้วรันสมุดภาพเล่มนี้ซ้ำเพื่อสร้าง AI ตัวใหม่ได้ทันที!